# Импорты

In [1]:
from enum import Enum
from tokenizers import Tokenizer
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tokenizers.models import BPE, Unigram, WordLevel, WordPiece
from torch.utils.data import DataLoader

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Инициализация токенизаторов

In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
tokenizer_1 = Tokenizer.from_pretrained(model_name_1)
print(f'Длина словаря {model_name_1}: {tokenizer_1.get_vocab_size()}')
print(f'Модель токенизатора {model_name_1}: {tokenizer_1.model}')

Длина словаря LiquidAI/LFM2.5-230M: 64402
Модель токенизатора LiquidAI/LFM2.5-230M: BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<|pad|>":0, "<|startoftext|>":1, "<|endoftext|>":2, "<|fim_pre|>":3, "<|fim_mid|>":4, ...}, merges=[("Ċ", "Ċ"), ("Ċ", "ĊĊ"), ("ĊĊ", "Ċ"), ("Ċ", "ĊĊĊ"), ("ĊĊ", "ĊĊ"), ...])


In [3]:
model_name_2 = "xlnet/xlnet-base-cased"
tokenizer_2 = Tokenizer.from_pretrained(model_name_2)
print(f'Длина словаря {model_name_2}: {tokenizer_2.get_vocab_size()}')
print(f'Модель токенизатора {model_name_2}: {tokenizer_2.model}')

Длина словаря xlnet/xlnet-base-cased: 32000
Модель токенизатора xlnet/xlnet-base-cased: Unigram(unk_id=0, vocab=[("<unk>", 0), ("<s>", 0), ("</s>", 0), ("<cls>", 0), ("<sep>", 0), ...], byte_fallback=False)


# Инициализация моделей

In [4]:
tokenizer_model_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype="auto",
                                               device_map="auto")

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 132/132 [00:00<00:00, 15463.30it/s]


In [5]:
tokenizer_model_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype="auto",
                                               device_map="auto")

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 63997.60it/s]


# Предобработка данных

In [6]:
class TokenizerModel(Enum):
    BPE = "BPE"
    WordPiece = "WordPiece"
    Unigram = "Unigram"

In [7]:
exact_match_dict = {}
model_1_exact_tokens = []
model_2_exact_tokens = []

if isinstance(tokenizer_1.model, BPE):
    if isinstance(tokenizer_2.model, Unigram):
        vocab_1 = tokenizer_1.get_vocab()
        vocab_2 = tokenizer_2.get_vocab()
        for key in vocab_1.keys():
            key_candidate = key
            if key[0] == 'Ġ':
                key = key.replace('Ġ', '▁')
            if key in vocab_2.keys():
                exact_match_dict[vocab_1[key_candidate]] = vocab_2[key]
                model_1_exact_tokens.append(vocab_1[key_candidate])
                model_2_exact_tokens.append(vocab_2[key])

exact_match_list = [(token_1, token_2) for token_1, token_2 in exact_match_dict.items()]

In [8]:
embedding_layer_1 = model_1.get_input_embeddings()
embedding_layer_2 = model_2.get_input_embeddings()
dataset = []

for token_1, token_2 in exact_match_list:
    embed_1 = embedding_layer_1.weight[token_1]
    embed_2 = embedding_layer_2.weight[token_2]
    dataset.append((embed_1, embed_2))


# Построение пайплайна загрузки данных в модель

In [9]:
class Kind(Enum):
    train = "train"
    test = "test"

TRAIN_RATIO = 0.8

In [10]:
class ExactMatchDataset(Dataset):
    def __init__(self, dataset: list[tuple], train_ratio: float, kind: Kind):
        self.dataset = self._train_test_split(dataset=dataset, train_ratio=train_ratio, kind=kind)
    
    def _train_test_split(self, dataset: list[tuple], train_ratio: float, kind: str):
        upper_limit = int(len(dataset)*train_ratio)
        if kind == Kind.train:
            return dataset[:upper_limit]
        elif kind == Kind.test:
            return dataset[upper_limit:]
        else:
            raise ValueError("Выбран неправильный тип датасета: возможен только train или test")

    def __getitem__(self, idx):
        model_1_embed = self.dataset[idx][0] # input embed
        model_2_embed = self.dataset[idx][1] # output embed
        return model_1_embed, model_2_embed
    
    def __len__(self):
        return len(self.dataset)

In [53]:
train = ExactMatchDataset(dataset, TRAIN_RATIO, Kind.train)
test = ExactMatchDataset(dataset, TRAIN_RATIO, Kind.test)

train_dataloader = DataLoader(train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test, batch_size=32, shuffle=True)

In [54]:
import torch
import torch.nn as nn

def compute_loss(alpha, beta, y_predict, y):
    mse_loss = nn.MSELoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    target = torch.ones(3)
    return alpha*mse_loss(y_predict, y) + beta*cosine_loss(y_predict, y, target)

In [55]:
class EmbedProjector(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=input_dim, out_features=num_hidden)
        self.layer_1 = nn.Linear(in_features=num_hidden, out_features=output_dim)

    def forward(self, x):
        x = self.layer_1(x)
        x = nn.functional.relu(x)
        x = self.layer_2(x)
        return x

In [64]:
print(torch.cuda.is_available())

False


In [66]:
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyTorch path:", torch.__file__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Python: /home/misha/Desktop/knowledge_distillation/.venv/bin/python
PyTorch: 2.12.1+cu130
PyTorch path: /home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/torch/__init__.py
CUDA build: 13.0
CUDA available: False


In [62]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
model = EmbedProjector(len(train[2][0]), 2048, len(train[2][1])).to(device)

Using cpu device


In [58]:
import torch.optim as optim
optimizer = optim.Adam(params=model.parameters(), lr=0.01)

In [60]:
epoch = 2
model.train()

EmbedProjector(
  (layer_1): Linear(in_features=2048, out_features=768, bias=True)
)

In [61]:
for _ in range(epoch):
    for x_train, y_train in train_dataloader:
        print(x_train)
        predict = model(x_train)
        loss = compute_loss(predict, y_train)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

tensor([[-0.0144, -0.0072, -0.0272,  ..., -0.0264,  0.0327,  0.0128],
        [-0.0457,  0.0103,  0.0006,  ..., -0.0415, -0.0820,  0.0087],
        [-0.0311, -0.0220,  0.0037,  ..., -0.0505,  0.0005, -0.0154],
        ...,
        [ 0.0299, -0.0245, -0.0170,  ..., -0.0161,  0.0079, -0.0425],
        [-0.0430, -0.0034, -0.0111,  ...,  0.0265, -0.0240,  0.0344],
        [ 0.0337, -0.0294,  0.0106,  ..., -0.0562,  0.0354, -0.0349]],
       dtype=torch.bfloat16, grad_fn=<StackBackward0>)


RuntimeError: mat1 and mat2 must have the same dtype, but got BFloat16 and Float